Proszę zbudować model służący do analizy sentymentu tweetów zebranych w zbiorze
danych:
https://www.kaggle.com/datasets/austinreese/trump-tweets
(podobnymi metodami jak przedstawione na zajęciach). Następnie dla zbudowanego
modelu proszę przeprowadzić analizę SHAP i przeprowadzić dyskusję uzyskanych
wyników.

## Plan

### 1. Preprocessing
- usuniecie linkow, mentions

### 2. Automatyczne etykietowanie
- VADER

### 3. Reprezentacja tekstu
- TF-IDF
- SBERT / embeddings

### 4. Budowa modeli
Klasyfikacja sentymentu:
- positive
- neutral
- negative

### 5. SHAP
- interpretacja predykcji
- wpływ słów / cech

### 6. Analiza wyników
- porównanie modeli
- interpretowalność
- błędy i ograniczenia

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("austinreese/trump-tweets")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# wyswietlanie pelnej wartosci tweetow
pd.set_option('display.max_colwidth', None)

data = pd.read_csv(os.path.join(path, "realdonaldtrump.csv")) 
print("data shape:", data.shape)
data.head()

## Preprocessnig

In [ ]:
import re

def clean_text(text):
    #text = text.lower()

    # usuwanie URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # usuwanie @mentions
    text = re.sub(r"@\s*\w+", "", text)

    # usuwanie nadmiarowych spacji
    # text = re.sub(r"\s+", " ", text).strip()

    return text

tweets = data["content"]
cleaned_tweets = tweets.apply(clean_text)
cleaned_tweets.head()

In [ ]:
import nltk

nltk.download('vader_lexicon')

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

data = []
threshold = 0.25

for tweet in cleaned_tweets:
    score = sia.polarity_scores(tweet)
    sentiment_score = score["compound"]

    if sentiment_score >= threshold:
        sentiment_label = "positive"
    elif sentiment_score <= -threshold:
        sentiment_label = "negative"
    else:
        sentiment_label = "neutral"

    data.append({
        "tweet": tweet,
        "sentiment_score": sentiment_score,
        "sentiment_label": sentiment_label
    })

df = pd.DataFrame(data)

print(df.head())

In [ ]:
df["sentiment_label"].value_counts()

## Sprawdzenie jakosci etykiet

In [ ]:
negative_tweets = df[df["sentiment_label"] == "negative"]

negative_tweets.head(10)    

In [ ]:
positive_tweets = df[df["sentiment_label"] == "positive"]

positive_tweets.head(10)    

In [ ]:
neutral_tweets = df[df["sentiment_label"] == "neutral"]

neutral_tweets.head(10)    

## Reprezentacja tekstu - TF-IDF


można też embeddingi, mozemy przetestowac co wyjdzie lepiej

In [ ]:
X = df["tweet"]
y = df["sentiment_label"]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    #5000 najczesciej wystepujacych slow w tweetach
    max_features=5000,
    stop_words="english"
)

X_tfidf = vectorizer.fit_transform(X)